In [1]:
import os, fcntl, time
import numpy as np
from pynq import Overlay, allocate
from rtlsdr import RtlSdr

print('[1] Loading Overlay...')
ol = Overlay('./base.bit')
pAudio = ol.audio_codec_ctrl_0
dma = ol.rtl_sdr.axi_dma_0

print('[2] Configuring Audio Codec (ADAU1761)...')
pAudio.configure(sample_rate=48000, iic_index=1, uio_name='audio-codec-ctrl')

# Unmute Mixer3/4 and enable Headphone Out via I2C directly
I2C_SLAVE = 0x0703
fd = os.open('/dev/i2c-1', os.O_RDWR)
fcntl.ioctl(fd, I2C_SLAVE, 0x3B)
def write_reg(addr, val):
    os.write(fd, bytes([0x40, addr, val]))

write_reg(0x1C, 0x21)  # R22 Playback Mixer Left Control 0
write_reg(0x1E, 0x41)  # R24 Playback Mixer Right Control 0
vol = (60 << 2) | 0x03 # Volume 60/63 + Headphone Enable
write_reg(0x23, vol)   # R29 Headphone Left Volume
write_reg(0x24, vol)   # R30 Headphone Right Volume
os.close(fd)
print('    ADAU1761 Mixer & Headphone UNMUTED!')



[1] Loading Overlay...


/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt mm2s_introut not created: Could not find UIO device for interrupt pin for IRQ number 64
  warnings.warn('Interrupt {} not created: {}'.format(
/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt s2mm_introut not created: Could not find UIO device for interrupt pin for IRQ number 65
  warnings.warn('Interrupt {} not created: {}'.format(


[2] Configuring Audio Codec (ADAU1761)...
    ADAU1761 Mixer & Headphone UNMUTED!
[3] Testing 440 Hz Tone from FPGA hardware for 2 seconds...
    Tone finished!
[4] Starting RTL-SDR FM Reception at 99.75 MHz...
[5] Streaming 5 blocks (~4 seconds of radio) directly to headphones...
    Block 1/5 streamed to audio codec (samples range: -22374 to 26893)
    Block 2/5 streamed to audio codec (samples range: -13019 to 15590)
    Block 3/5 streamed to audio codec (samples range: -21055 to 24056)
    Block 4/5 streamed to audio codec (samples range: -20144 to 27408)
    Block 5/5 streamed to audio codec (samples range: -23627 to 25539)
[6] Done testing!


In [2]:
# Step A: 2-second test tone
print('[3] Testing 440 Hz Tone from FPGA hardware for 2 seconds...')
pAudio.mmio.write(0x20, 1)
time.sleep(2.0)
pAudio.mmio.write(0x20, 0)
print('    Tone finished!')

[3] Testing 440 Hz Tone from FPGA hardware for 2 seconds...
    Tone finished!


In [3]:


# Step B: RTL-SDR FM stream test
print('[4] Starting RTL-SDR FM Reception at 99.75 MHz...')
sdr = RtlSdr()
sdr.sample_rate = 1920000
sdr.center_freq = 99750000
sdr.gain = 'auto'

len_in = 1024 * 1500
len_out = int(len_in / 40)
input_buffer = allocate(shape=(len_in,), dtype=np.uint16)
output_buffer = allocate(shape=(len_out,), dtype=np.int16)

# Read 5 blocks and stream into DMA -> Codec
print('[5] Streaming 5 blocks (~4 seconds of radio) directly to headphones...')
for b in range(5):
    data = sdr.read_bytes(len_in * 2)
    raw_uint8 = np.frombuffer(data, dtype=np.uint8)
    raw_int8 = (raw_uint8.astype(np.int16) - 128).astype(np.int8)
    packed = raw_int8.view(np.uint16)
    np.copyto(input_buffer, packed)
    
    dma.sendchannel.transfer(input_buffer)
    dma.recvchannel.transfer(output_buffer)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    print(f'    Block {b+1}/5 streamed to audio codec (samples range: {np.min(output_buffer)} to {np.max(output_buffer)})')

sdr.close()
del input_buffer, output_buffer
print('[6] Done testing!')


[4] Starting RTL-SDR FM Reception at 99.75 MHz...
[5] Streaming 5 blocks (~4 seconds of radio) directly to headphones...
    Block 1/5 streamed to audio codec (samples range: -21899 to 23618)
    Block 2/5 streamed to audio codec (samples range: -20699 to 24019)
    Block 3/5 streamed to audio codec (samples range: -21275 to 23813)
    Block 4/5 streamed to audio codec (samples range: -21030 to 24744)
    Block 5/5 streamed to audio codec (samples range: -22025 to 23578)
[6] Done testing!
